## EJERCICIO 1


In [78]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql.functions import min, max, avg, countDistinct, col, mean, when,row_number,collect_list,unix_timestamp, from_unixtime, month, year, split, to_timestamp, udf,sum as _sum, hour,date_format
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType,TimestampType
import seaborn as sns

In [79]:
#Cargar los datos en un DataFrame de Spark infiriendo el esquema.
spark = SparkSession.builder.appName("Ejercicio1").getOrCreate()
df = spark.read.csv("Turismo.csv", header=True, inferSchema=True,sep=";")


In [80]:
#Mostrar el esquema del DataFrame y los primeros 5 registros.
df.printSchema()
df.show(5)
   

root
 |-- ID: string (nullable = true)
 |-- VISITA: string (nullable = true)
 |-- MES: string (nullable = true)
 |-- FECHA: string (nullable = true)
 |-- SATISFACCION_INFOR: integer (nullable = true)
 |-- SATISFACCION_RECORRIDO: integer (nullable = true)
 |-- SATISFACCION_GUIA: integer (nullable = true)
 |-- SATISFACCION_GLOBAL: integer (nullable = true)
 |-- LO_MEJOR: string (nullable = true)
 |-- LO_PEOR: string (nullable = true)
 |-- OPINION: string (nullable = true)
 |-- SUGERENCIAS: string (nullable = true)

+---+--------------------+----------+----------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+
| ID|              VISITA|       MES|     FECHA|SATISFACCION_INFOR|SATISFACCION_RECORRIDO|SATISFACCION_GUIA|SATISFACCION_GLOBAL|            LO_MEJOR|             LO_PEOR|             OPINION|         SUGERENCIAS|
+---+--------------------+----------+----------+-------

In [81]:
#Convertir la columna "FECHA" en formato de fecha (dd/MM/yyyy) y mostrar el esquema.
df = df.withColumn("FECHA", to_timestamp(df["FECHA"], "dd/MM/yyyy"))
df.printSchema()
df.show(5)

   

root
 |-- ID: string (nullable = true)
 |-- VISITA: string (nullable = true)
 |-- MES: string (nullable = true)
 |-- FECHA: timestamp (nullable = true)
 |-- SATISFACCION_INFOR: integer (nullable = true)
 |-- SATISFACCION_RECORRIDO: integer (nullable = true)
 |-- SATISFACCION_GUIA: integer (nullable = true)
 |-- SATISFACCION_GLOBAL: integer (nullable = true)
 |-- LO_MEJOR: string (nullable = true)
 |-- LO_PEOR: string (nullable = true)
 |-- OPINION: string (nullable = true)
 |-- SUGERENCIAS: string (nullable = true)

+---+--------------------+----------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+
| ID|              VISITA|       MES|              FECHA|SATISFACCION_INFOR|SATISFACCION_RECORRIDO|SATISFACCION_GUIA|SATISFACCION_GLOBAL|            LO_MEJOR|             LO_PEOR|             OPINION|         SUGERENCIAS|
+---+--------------------+--------

In [82]:
#Listar las 3 visitas con la mayor puntuación de: SATISFACCION_GLOBAL.
df.orderBy(col("SATISFACCION_GLOBAL").desc()).show(3)

+---+--------------------+---------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+-------+--------------------+--------------------+
| ID|              VISITA|      MES|              FECHA|SATISFACCION_INFOR|SATISFACCION_RECORRIDO|SATISFACCION_GUIA|SATISFACCION_GLOBAL|            LO_MEJOR|LO_PEOR|             OPINION|         SUGERENCIAS|
+---+--------------------+---------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+-------+--------------------+--------------------+
| 15|LA HISTORIA IMPRE...|   AGOSTO|2023-08-11 00:00:00|                10|                    10|               10|                 10|                NULL|   NULL|       Como esperaba|Amplia nuevas vis...|
|  3|    MADRID HISTÓRICO|    MARZO|2023-03-03 00:00:00|                10|                    10|               10|                 10|El conjunto está ...|   NULL|Mej

In [83]:
#Contar cuántas visitas se realizaron en cada mes del año y mostrarlas en orden descendente.
df.groupBy(month("FECHA").alias("MES")).count().orderBy(col("count").desc()).show()

+----+-----+
| MES|count|
+----+-----+
|  10|   41|
|   9|   36|
|   4|   34|
|  12|   32|
|   8|   31|
|  11|   29|
|   7|   26|
|   6|   21|
|   2|   18|
|   5|   14|
|   1|   13|
|NULL|    7|
|   3|    3|
+----+-----+



In [84]:
#Filtrar las visitas realizadas en septiembre y mostrar sólo los campos: VISITA y SATISFACCION_GLOBAL.
df.filter(month("FECHA") == 9).select("VISITA", "SATISFACCION_GLOBAL").show()
   

+--------------------+-------------------+
|              VISITA|SATISFACCION_GLOBAL|
+--------------------+-------------------+
|  MADRID EN FEMENINO|                 10|
|LA LATINA Y EL MU...|                 10|
|  MADRID EN FEMENINO|                  9|
|   MADRID MONUMENTAL|                 10|
|   MADRID MONUMENTAL|                 10|
|  MADRID EN FEMENINO|                  6|
|  MADRID EN FEMENINO|                  6|
|LA HISTORIA IMPRE...|                 10|
|LA LATINA Y EL MU...|                  9|
|  MADRID EN FEMENINO|                  8|
|LA HISTORIA IMPRE...|                 10|
|   MADRID MONUMENTAL|                 10|
|  MADRID EN FEMENINO|                  8|
|HISTORIAS Y LEYEN...|                  9|
|HISTORIAS Y LEYEN...|                 10|
|LA NUEVA PLAZA DE...|                 10|
|LA NUEVA PLAZA DE...|                 10|
|LA NUEVA PLAZA DE...|                 10|
|LA NUEVA PLAZA DE...|                 10|
|LA LATINA Y EL MU...|                 10|
+----------

In [85]:
 #Reemplazar valores nulos en la columna: LO_PEOR con el valor: "Sin comentarios".
df = df.withColumn("LO_PEOR", when(col("LO_PEOR").isNull(), "Sin comentarios").otherwise(col("LO_PEOR")))
df.show()
    

+---+--------------------+----------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+
| ID|              VISITA|       MES|              FECHA|SATISFACCION_INFOR|SATISFACCION_RECORRIDO|SATISFACCION_GUIA|SATISFACCION_GLOBAL|            LO_MEJOR|             LO_PEOR|             OPINION|         SUGERENCIAS|
+---+--------------------+----------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+
| 12|LEYENDAS, CASAS E...| DICIEMBRE|2023-12-17 00:00:00|                10|                    10|               10|                 10|        7 chimeneas |                Nada|Mejor de lo que e...|Nada, todo perfecto |
| 19|  MADRID EN FEMENINO|    AGOSTO|2023-08-12 00:00:00|                 8|                     8|             

In [ ]:
#Crear una nueva columna llamada SATISFACCION_PROMEDIO que sea el promedio de las 3 categorías de satisfacción: SATISFACCION_INFOR, SATISFACCION_RECORRIDO, SATISFACCION_GUIA. Debes crear tu propia función de usuario y registrarla como una función de Spark. Con SQL y sin SQL.

def promedio(s1, s2, s3):
    valores = [s1, s2, s3]
    valores = [v for v in valores if v is not None]
    return sum(valores) / len(valores) if valores else None  

promedio_udf = udf(promedio, DoubleType())

df = df.withColumn(
    "SATISFACCION_PROMEDIO",
    promedio_udf(col("SATISFACCION_INFOR"), col("SATISFACCION_RECORRIDO"), col("SATISFACCION_GUIA"))
)

df.show()


#df.createOrReplaceTempView("turismo")
#spark.udf.register("promedio", promedio, DoubleType())
#df = spark.sql("SELECT *, promedio(SATISFACCION_INFOR, SATISFACCION_RECORRIDO, SATISFACCION_GUIA) AS SATISFACCION_PROMEDIO FROM turismo")
#df.show()

+---+--------------------+----------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------------------+
| ID|              VISITA|       MES|              FECHA|SATISFACCION_INFOR|SATISFACCION_RECORRIDO|SATISFACCION_GUIA|SATISFACCION_GLOBAL|            LO_MEJOR|             LO_PEOR|             OPINION|         SUGERENCIAS|SATISFACCION_PROMEDIO|
+---+--------------------+----------+-------------------+------------------+----------------------+-----------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------------------+
| 12|LEYENDAS, CASAS E...| DICIEMBRE|2023-12-17 00:00:00|                10|                    10|               10|                 10|        7 chimeneas |                Nada|Mejor de lo que e...|Nada, todo perfecto |                 10.0|
| 19|  MADRID EN FEMENIN

In [87]:
#Guardar el resultado final en formato .csv de tal forma que si el archivo ya existe que lo sobreescriba.
df.write.mode("overwrite").csv("turismo_resultado.csv", header=True)

## EJERCICIO 2


In [62]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("Ventana fija IABD WordCount") \
        .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4") \
        .master("local[*]") \
        .config("spark.streaming.stopGracefullyOnShutdown", "true") \
        .config("spark.sql.shuffle.partitions", 3) \
        .getOrCreate()

dfLineas = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .option('includeTimestamp', 'true')\
    .load()

25/03/10 16:48:45 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.


In [63]:
from pyspark.sql.functions import explode, split
dfPalabras = dfLineas.select(
    explode(split(dfLineas.value, ' ')).alias('palabra'),
    dfLineas.timestamp)


In [64]:
#Ventana fija

from pyspark.sql.functions import window
windowedCounts = dfPalabras.groupBy(
    window(dfPalabras.timestamp, "2 minutes"), dfPalabras.palabra
).count().orderBy('window')

In [66]:
palabrasQuery = windowedCounts.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option('truncate', 'false')\
    .start()

25/03/10 16:48:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-7cb35d3c-66a7-4048-a36f-b66641e02404. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/03/10 16:48:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------+-------+-----+
|window|palabra|count|
+------+-------+-----+
+------+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-------+-----+
|window                                    |palabra|count|
+------------------------------------------+-------+-----+
|{2025-03-10 16:48:00, 2025-03-10 16:50:00}|       |1    |
+------------------------------------------+-------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+-------+-----+
|window                                    |palabra|count|
+------------------------------------------+-------+-----+
|{2025-03-10 16:48:00, 2025-03-10 16:50:00}|tardes |1    |
|{2025-03-10 16:48:00, 2025-03-10 16:50:00}|buenas |1    |
|{2025-03-10 16:48:00, 20

25/03/10 16:52:13 WARN TextSocketMicroBatchStream: Stream closed by localhost:9999
